<a href="https://colab.research.google.com/github/SaiSanthosh1508/Foundation-Models-From-Scratch/blob/main/LoRA_%26_DoRA_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import numpy as np
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.nn as nn
import torch

In [ ]:
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

## Setting and Dataset

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

train_dataset = datasets.MNIST(root='data/mnist/',
                               train=True,
                               transform=transforms.ToTensor(),
                               download=True)

test_dataset = datasets.MNIST(root='data/mnist/',
                              train=False,
                              transform=transforms.ToTensor())

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

for images,labels in train_loader:
  print('Image batch dimensions:', images.shape)
  print('Image label dimensions:', labels.shape)
  break

## Multi Layer Perceptron Model

In [ ]:
random_seed = 123
lr = 0.005
num_epochs = 3

num_features = 784
num_hidden_1 = 128
num_hidden_2 = 256
num_classes = 10

class MultilayerPerceptron(nn.Module):

  def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
    super().__init__()

    self.layers = nn.Sequential(
        nn.Linear(num_features, num_hidden_1),
        nn.ReLU(),
        nn.Linear(num_hidden_1, num_hidden_2),
        nn.ReLU(),
        nn.Linear(num_hidden_2, num_classes)
    )

  def forward(self, x):
    return self.layers(x)

In [ ]:
torch.manual_seed(random_seed)

model_pretrained = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes
)

model_pretrained.to(DEVICE)
optimizer_pretrained = torch.optim.Adam(model_pretrained.parameters(), lr=lr)

In [ ]:
def compute_accuracy(model, data_loader, device):
  model.eval()
  correct_pred, total = 0,0

  with torch.no_grad():
    for images,labels in data_loader:
      images = images.reshape(-1,28*28).to(device)
      targets = labels.to(device)

      logits = model(images)

      _, predicted_labels = torch.max(logits, 1)
      total += targets.size(0)
      correct_pred += (predicted_labels == targets).sum()

    return correct_pred.float() / total * 100

In [ ]:
def train(num_epochs, model, optimizer, train_loader, device):

    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):

            features = features.view(-1, 28*28).to(device)
            targets = targets.to(device)

            # FORWARD AND BACK PROP
            logits = model(features)
            loss = F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()

            # UPDATE MODEL PARAMETERS
            optimizer.step()

            # LOGGING
            if not batch_idx % 400:
                print('Epoch: %03d/%03d | Batch %03d/%03d | Loss: %.4f'
                      % (epoch+1, num_epochs, batch_idx,
                          len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (
                  epoch+1, num_epochs,
                  compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))

    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

In [ ]:
train(num_epochs, model_pretrained, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model_pretrained, test_loader, DEVICE):.2f}%')

## Multi Layer Perceptron with LoRA

In [ ]:
class LoRALayer(nn.Module):

  def __init__(self, in_dim, out_dim, rank, alpha):
    super().__init__()
    self.A = nn.Linear(in_dim, rank, bias=False)
    self.B = nn.Linear(rank, out_dim, bias=False)
    self.scale = alpha / rank

    nn.init.kaiming_normal_(self.A.weight)
    nn.init.zeros_(self.B.weight)

  def forward(self, x):
    x = self.A(x)
    x = self.B(x)
    return x * self.scale

In [ ]:
class LinearWithLoRA(nn.Module):
  def __init__(self, linear, rank, alpha):
    super().__init__()

    self.layer = linear

    self.lora_layer = LoRALayer(
        linear.in_features,
        linear.out_features,
        rank = rank,
        alpha = alpha
    )

  def forward(self, x):
    linear_output = self.layer(x)
    lora_output = self.lora_layer(x)

    return linear_output + lora_output

In [ ]:
import copy

# Copy the pretrained model architecture
model_lora = copy.deepcopy(model_pretrained)

print(model_lora)
print(f"Total number of trainable parameters: {sum(p.numel() for p in model_lora.parameters() if p.requires_grad)}")

In [ ]:
def replace_linear_layers(model):

  for name,child in model.named_children():
    if isinstance(child, nn.Linear):
      lora_layer = LinearWithLoRA(
          linear = child,
          rank = 16,
          alpha = 16
      )
      setattr(model, name, lora_layer)
    else:
      replace_linear_layers(child)

replace_linear_layers(model_lora)

In [ ]:
model_lora

In [ ]:
for name, param in model_lora.named_parameters():
  param.requires_grad = False

for name, param in model_lora.named_parameters():
  if "lora" in name:
    param.requires_grad = True

In [ ]:
print(f"Total number of trainable parameters: {sum(p.numel() for p in model_lora.parameters() if p.requires_grad)}")

In [ ]:
model_lora.to(DEVICE)
print(f"Accuracy before training: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}")

In [ ]:
optimizer_lora = torch.optim.Adam(model_lora.parameters(),lr=lr)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f"Accuracy after training: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}")

## Multi Layer Perceptron with DoRA

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = nn.Linear(in_dim, rank, bias=False)
        self.B = nn.Linear(rank, out_dim, bias=False)
        self.scale = alpha / rank

        nn.init.kaiming_normal_(self.A.weight)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        x = self.A(x)
        x = self.B(x)
        return x * self.scale

class LinearWithDoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )
        self.m = nn.Parameter(
            self.linear.weight.norm(p=2, dim=1, keepdim=True)
        )

    def forward(self, x):
        lora_weight = self.lora.B.weight @ self.lora.A.weight

        numerator = self.linear.weight + (lora_weight * self.lora.scale)

        denominator = numerator.norm(p=2, dim=1, keepdim=True)
        directional_component = numerator / (denominator + 1e-6)

        new_weight = self.m * directional_component

        return F.linear(x, new_weight, self.linear.bias)

In [ ]:
model_dora = copy.deepcopy(model_pretrained)
model_dora

In [ ]:
print(f"Total number of trainable parameters: {sum(p.numel() for p in model_dora.parameters())}")

In [ ]:
def replace_linear_layers_dora(model):

  for name,child in model.named_children():
    if isinstance(child, nn.Linear):
      lora_layer = LinearWithDoRAMerged(
          linear = child,
          rank = 16,
          alpha = 16)
      setattr(model, name, lora_layer)
    else:
      replace_linear_layers_dora(child)

replace_linear_layers_dora(model_dora)

In [ ]:
model_dora

In [ ]:
for name, param in model_dora.named_parameters():
  param.requires_grad = False

for name, param in model_dora.named_parameters():
  if "lora" in name or name.endswith(".m"):
    param.requires_grad = True

In [ ]:
print(f"Total number of trainable parameters: {sum(p.numel() for p in model_dora.parameters() if p.requires_grad)}")

In [ ]:
model_dora.to(DEVICE)
optimizer_dora = torch.optim.Adam(model_dora.parameters(),lr=lr)
train(num_epochs, model_lora, optimizer_dora, train_loader, DEVICE)
print(f"Accuracy after training: {compute_accuracy(model_dora, test_loader, DEVICE):.2f}")